In [1]:
#子任务 1：GPT-2 124M + LoRA 微调
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model

DATA_PATH = "./data_tiny_shakespeare/input.txt"

#硬件与基础配置 
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

#加载数据集
with open(DATA_PATH, "r", encoding="utf-8") as f:
    text = f.read()

dataset = DatasetDict({
    "train": Dataset.from_dict({"text": [text]})
})

#数据预处理 
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token  # GPT2原生无pad token，用结束符代替

def preprocess_function(examples):
    # 拼接文本，截断到512长度
    concatenated_examples = [k for k in examples["text"]]
    tokenized = tokenizer(
        concatenated_examples,
        truncation=True,
        max_length=512,
        return_overflowing_tokens=False,
    )
    return tokenized

tokenized_datasets = dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

# 数据整理器：因果语言模型，不需要MLM
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

#加载模型 + 配置LoRA
model = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    #torch_dtype=torch.float16,
).to(device)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    target_modules=["c_attn"],  # GPT2注意力层的合并线性层
    lora_dropout=0.05,
    fan_in_fan_out=True
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

#训练参数（针对4060 8G优化）
training_args = TrainingArguments(
    output_dir="./gpt2-lora-shakespeare",
    learning_rate=2e-4,
    per_device_train_batch_size=2,  # 批次大小2
    gradient_accumulation_steps=4,  # 梯度累积4步，等效batch=8
    num_train_epochs=10,
    weight_decay=0.01,
    save_strategy="epoch",
    #fp16=True,
    logging_steps=2,
    remove_unused_columns=False,
    report_to="none",  # 关闭wandb等日志，避免额外开销
)

2026-03-27 22:06:19.063651: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-27 22:06:19.063726: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-27 22:06:19.064744: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


使用设备: cuda


Map:   0%|          | 0/1 [00:00<?, ? examples/s]

trainable params: 2,359,296 || all params: 126,799,104 || trainable%: 1.8607


In [2]:
#开始训练 
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

trainer.train()

Step,Training Loss
2,1.082900
4,1.053600
6,1.063100
8,1.047100
10,1.021600


TrainOutput(global_step=10, training_loss=1.0536792993545532, metrics={'train_runtime': 10.6706, 'train_samples_per_second': 0.937, 'train_steps_per_second': 0.937, 'total_flos': 2685397893120.0, 'train_loss': 1.0536792993545532, 'epoch': 10.0})

In [3]:
#文本生成测试
def generate_text(prompt, max_length=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_length,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

TITLE="Hello world!"
print("文本续写开头:"+TITLE)
print("文本续写内容:\n"+generate_text(TITLE))

文本续写开头:Hello world!
文本续写内容:
Hello world!
That's right, that is why we're here. Because when you look at your surroundings - where do the lights go? What does it mean to be alive in a way and not die yet?"


In [4]:
#子任务 2：GPT-2 124M + PT 微调
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import PromptTuningConfig, PromptTuningInit, get_peft_model

#加载完全干净的原始模型
model_PT = GPT2LMHeadModel.from_pretrained(
    "gpt2",
    #torch_dtype=torch.float16,
    #use_cache=False  # 训练时关闭cache省显存
).to(device)

prompt_config = PromptTuningConfig(
    task_type="CAUSAL_LM",  # 必须明确指定为因果语言模型
    # prompt_tuning_init=PromptTuningInit.RANDOM,  # 生成任务推荐随机初始化
    prompt_tuning_init=PromptTuningInit.TEXT,  # 改为文本初始化
    prompt_tuning_init_text="From fairest creatures we desire increase,",  # 莎士比亚风格初始化文本
    num_virtual_tokens=15,  # 软提示长度hzy
    tokenizer_name_or_path="gpt2",  # 必须指定分词器，用于初始化软提示
)

model_PT = get_peft_model(model_PT, prompt_config)
model_PT.print_trainable_parameters() 

training_args_PT = TrainingArguments(
    output_dir="./gpt2-pt-shakespeare",
    learning_rate=3e-3,  # PT专属高学习率（关键！）
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=10,  # 增加epoch，让PT充分收敛hzy
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=2,
    remove_unused_columns=False,
    report_to="none",
)

trainer_PT = Trainer(
    model=model_PT,
    args=training_args_PT,
    train_dataset=tokenized_datasets["train"],
    data_collator=data_collator,
)

trainer_PT.train()

trainable params: 11,520 || all params: 124,451,328 || trainable%: 0.0093


Step,Training Loss
2,1.123100
4,1.111700
6,1.091800
8,1.082400
10,1.070500


TrainOutput(global_step=10, training_loss=1.095910334587097, metrics={'train_runtime': 10.8516, 'train_samples_per_second': 0.922, 'train_steps_per_second': 0.922, 'total_flos': 2612920320000.0, 'train_loss': 1.095910334587097, 'epoch': 10.0})

In [5]:
#生成测试
def generate_text_PT(prompt, max_length=100):
    model_PT.eval()  # 切换到评估模式
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model_PT.generate(
            **inputs,
            max_length=max_length,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

TITLE_PT="Hello world!"
print("文本续写开头:"+TITLE_PT)
print("文本续写内容:\n"+generate_text_PT(TITLE_PT))

文本续写开头:Hello world!
文本续写内容:
Hello world! I hope you like it.


/home/hexiaoya/miniconda3/envs/ai/lib/python3.10/site-packages/peft/peft_model.py:2141: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")
